# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): +10.0,  # Entrega
            (2, 2): +2.0,  # Carga
            (3, 5): -10.0,  # Peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        # Si es terminal, el estado es absorbente
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Determinar probabilidades según el tipo de piso
        if state in self.slippery_states:
            p_intended = 0.60
            p_perp = 0.20
        else:
            p_intended = 0.90
            p_perp = 0.05

        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perp),
            (perp2, p_perp),
        ]

        transitions = []
        for act, prob in outcomes:
            next_state = (state[0] + act[0], state[1] + act[1])

            # Si golpea una pared o sale del mapa, permanece en el mismo estado
            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        return transitions



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = (
                    grid.get_reward(state) + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change, abs(V_new[state] - V[state])
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V),
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(
                biggest_change, abs(V_new[state] - V[state])
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V),
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # Política inicial arbitraria (DOWN)
    initial_action = (1, 0)
    policy = {
        state: initial_action
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        V, eval_iterations = policy_evaluation(
            grid, policy, threshold=threshold
        )
        new_policy = policy_improvement(grid, V)

        changed = sum(
            new_policy[state] != policy[state] for state in new_policy
        )

        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy
        if changed == 0:
            break

    V, _ = policy_evaluation(grid, policy, threshold=threshold)
    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 95, 'changed_actions': 17}, {'policy_iteration': 2, 'evaluation_sweeps': 89, 'changed_actions': 10}, {'policy_iteration': 3, 'evaluation_sweeps': 77, 'changed_actions': 5}, {'policy_iteration': 4, 'evaluation_sweeps': 19, 'changed_actions': 1}, {'policy_iteration': 5, 'evaluation_sweeps': 22, 'changed_actions': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

RESPUESTA: Desde START (0,0), el robot prefiere ir a la estación de carga +2 (2,2).

2. ¿Por qué una recompensa menor podría ser óptima?

RESPUESTA: Debido al costo por paso (-1) y al factor de descuento (0.9$). Llegar a la carga toma muy pocos pasos, mientras que viajar hasta la entrega (+10) en (0,5) requiere atravesar un camino largo, peligroso y rodeado de casillas resbalosas, donde el acumulado de castigos por paso reduce el valor esperado descontado.

3. ¿En qué estados el piso resbaloso cambia la decisión?

RESPUESTA: En las casillas adyacentes a las trampas mortales o peligros (por ejemplo, cerca de (3,5) o (1,4)). En esos puntos, el riesgo de desviarse por resbalar hace que la política elija caminos rodeados de paredes o paredes que actúan como "frenos de seguridad".

4. ¿Qué papel cumple el costo por paso `-1`?

RESPUESTA: Actúa como una penalización por tiempo que presiona al agente a terminar el episodio lo antes posible. Un costo alto por paso hace que metas cercanas pequeñas (+2) resulten más atractivas que metas lejanas grandes (+10).

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

RESPUESTA: Porque la estocasticidad del entorno varía según el tipo de superficie en la que se encuentra el agente: el piso normal mantiene una probabilidad de éxito del 90%, mientras que el piso resbaloso reduce la certeza al 60% y aumenta las desviaciones al 20%.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

PREDICCIÓN: El costo de viajar se reduce 10 veces. Desde START, el robot cambiará de decisión y buscará la entrega de +10 en (0,5), ya que la penalización por dar pasos adicionales no destruye el beneficio de la recompensa mayor.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

EFECTO: Al aumentar la incertidumbre al 30% por cada lado, la política se vuelve sumamente cautelosa, alejándose de las zonas con peligro mortal (-10) e incentivando al agente a tomar rutas donde las paredes actúen como contención.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

EFECTO: El agente valora casi por igual las recompensas presentes y futuras. Esto hace que la recompensa distante de +10 sea dominante, por lo que la política desde START priorizará ir hasta la entrega.

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.

El cambio de política en START entre ir a la carga (+2) o ir a la entrega (+10) ocurre aproximadamente cuando living_reward se sitúa alrededor de -0.35 a -0.40.
